# EDA - Telco Customer Churn

## 1. Overview

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
%matplotlib inline

df = pd.read_csv("../data/cleaned.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns\n")
print("Columns and dtypes:")
print(df.dtypes.to_string())
print()
df.head()

## 2. Target Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
counts = df["Churn"].value_counts()
pcts = df["Churn"].value_counts(normalize=True) * 100

bars = ax.bar(
    counts.index, counts.values,
    color=["#2ecc71", "#e74c3c"], edgecolor="white", width=0.5
)

for bar, pct in zip(bars, pcts):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 30,
        f"{bar.get_height():,}\n({pct:.1f}%)",
        ha="center", va="bottom", fontsize=12, fontweight="bold"
    )

ax.set_title("Churn Class Distribution", fontsize=16, fontweight="bold", pad=15)
ax.set_xlabel("Churn", fontsize=13)
ax.set_ylabel("Count", fontsize=13)
ax.set_ylim(0, counts.max() * 1.2)
plt.tight_layout()
plt.show()

## 3. Missing Values

In [ ]:
null_counts = df.isnull().sum().to_frame(name="null_count")

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(
    null_counts.T,
    annot=True, fmt="d", cmap="Reds",
    linewidths=0.5, ax=ax, cbar_kws={"label": "Null Count"}
)
ax.set_title("Missing Values per Column", fontsize=15, fontweight="bold", pad=12)
ax.set_xlabel("Column", fontsize=12)
ax.set_ylabel("")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 4. Feature Distributions

In [ ]:
numeric_cols = list(df.select_dtypes(include="number").columns)
n_cols = 3
n_rows = -(-len(numeric_cols) // n_cols)  # ceiling division

fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].hist(df[col].dropna(), bins=30, color="#3498db", edgecolor="white", alpha=0.85)
    axes[i].set_title(col, fontsize=11, fontweight="bold")
    axes[i].set_xlabel(col, fontsize=9)
    axes[i].set_ylabel("Count", fontsize=9)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Feature Distributions (Numeric)", fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## 5. Correlation Matrix

In [ ]:
corr = df.select_dtypes(include="number").corr()

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(
    corr, annot=True, fmt=".2f", cmap="coolwarm",
    center=0, linewidths=0.5, square=True,
    ax=ax, cbar_kws={"shrink": 0.8}
)
ax.set_title("Correlation Matrix - Numeric Features", fontsize=15, fontweight="bold", pad=12)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 6. Features vs Target

In [ ]:
top_features = ["tenure", "MonthlyCharges", "TotalCharges"]

fig, axes = plt.subplots(1, 3, figsize=(14, 6))

for ax, feat in zip(axes, top_features):
    sns.boxplot(
        data=df, x="Churn", y=feat,
        palette={"No": "#2ecc71", "Yes": "#e74c3c"}, ax=ax
    )
    ax.set_title(f"{feat} vs Churn", fontsize=12, fontweight="bold")
    ax.set_xlabel("Churn", fontsize=11)
    ax.set_ylabel(feat, fontsize=11)

fig.suptitle("Top Numeric Features vs Churn", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 7. Key Findings

- **Churn rate ~26%**: The dataset is moderately imbalanced -- roughly 1 in 4 customers churns. Models should apply class weighting or oversampling.
- **Tenure is the strongest predictor**: Churned customers have significantly lower median tenure (~10 months vs ~38 months), indicating early-lifecycle customers are highest risk.
- **Higher monthly charges correlate with churn**: Churners pay more per month on average, linked to month-to-month contracts and premium service bundles.
- **TotalCharges and tenure are highly correlated**: Strong positive correlation (~0.83) makes them near-redundant; consider dropping TotalCharges to reduce multicollinearity.
- **SeniorCitizen is a minority but at-risk group**: ~16% of customers are seniors, yet they churn at a disproportionately higher rate -- a segment worth targeted retention efforts.